# CURE-Rec — complete notebook execution

This notebook is the **single entry point** for the complete Milestone 1 workflow. Running it in order performs every implemented stage:

1. fetch/load external recommendation data;
2. standardize and audit the evidence level;
3. run all registered CPU recommender models;
4. generate external-data analysis assets;
5. execute all CURE-Sim scenarios and all 64 intervention coalitions;
6. compute exact Shapley values, interaction regions, feasibility sensitivity, and direct robust policy selection;
7. generate every numbered CURE-Sim paper asset, logs, manifests, and decision card.

The external-data stage tests data and recommender-model logic. The CURE-Sim stage is the oracle causal benchmark; the notebook does not misrepresent ordinary ratings data as long-horizon policy-intervention evidence.

## 1. Setup

Install once from `paper-ideas/CURE-Rec/code/`:

```bash
python3 -m venv .venv
source .venv/bin/activate
python -m pip install -e '.[dev]'
jupyter lab notebooks/00_cure_rec_quickstart.ipynb
```

In [ ]:
from pathlib import Path
import json
import sys
import pandas as pd

CWD = Path.cwd().resolve()
CANDIDATES = [CWD, CWD / 'paper-ideas' / 'CURE-Rec' / 'code', *CWD.parents]
ROOT = next((p for p in CANDIDATES if (p / 'pyproject.toml').exists() and (p / 'cure_rec').exists()), None)
if ROOT is None:
    raise RuntimeError('Open the notebook from the CURE-Rec code directory or repository root.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from cure_rec.config import load_settings
from cure_rec.workflow import run_full_workflow

print('Project root:', ROOT)


## 2. Configure the entire run

`quick` is designed for interactive use. `full` uses the larger CURE-Sim configuration. The external-data fetch is explicit and visible: set `FETCH_IF_MISSING = False` when using already downloaded local data.

In [ ]:
# CURE-Sim configuration
CURE_MODE = 'quick'  # quick | full
config_name = 'curesim_quickstart.yaml' if CURE_MODE == 'quick' else 'curesim_full.yaml'
settings = load_settings(ROOT / 'configs' / config_name)

# External-data and registered model configuration
PUBLIC_DATASET = 'movielens_1m'  # movielens_1m | coat | yahoo_r3 | csv
PUBLIC_SOURCE = ROOT / 'data' / 'raw' / PUBLIC_DATASET
FETCH_IF_MISSING = True  # explicit opt-in network retrieval for MovieLens-1M / Coat
RUN_BPR_MF = True
BPR_UPDATES = 50_000 if CURE_MODE == 'quick' else 200_000
MAX_EVAL_USERS = 1_000

print('CURE config:', config_name, '| hash:', settings.config_hash())
print('CURE users/items/horizon:', settings.simulator.n_users, settings.simulator.n_items, settings.simulator.horizon)
print('CURE interventions:', list(settings.interventions.costs))
print('CURE scenarios:', [scenario.name for scenario in settings.scenarios])
print('External dataset:', PUBLIC_DATASET, '| source:', PUBLIC_SOURCE)


## 3. Run every implemented stage

This one cell calls the full orchestrator. It **fetches/loads data first**, audits the evidence, runs popularity and BPR-MF baselines when chronology is available, and then runs the full CURE-Sim causal workflow and asset generator.

In [ ]:
workflow = run_full_workflow(
    settings,
    dataset=PUBLIC_DATASET,
    source=PUBLIC_SOURCE,
    download=FETCH_IF_MISSING,
    run_bpr=RUN_BPR_MF,
    bpr_updates=BPR_UPDATES,
    max_eval_users=MAX_EVAL_USERS,
)

public_result = workflow.dataset
data_analysis = workflow.analysis
logger = workflow.logger
game = workflow.game
RUN_DIR = workflow.cure_run_dir
decision = workflow.decision

print('External-data analysis run:', data_analysis.run_dir)
print('External-data evidence level:', data_analysis.audit.permitted_claim)
print('CURE-Rec run:', RUN_DIR)
print('Decision:', decision.action)
print('Selected portfolio:', decision.selected_interventions)
print('Worst-case improvement:', round(decision.lower_improvement, 5))


## 4. Inspect loaded data, audit logic, and every registered baseline model

These results describe the external interaction data. They are intentionally separate from the CURE-Sim causal results below.

In [ ]:
print('Loader metadata:')
print(public_result.metadata)
print('Audit notes:', *data_analysis.audit.notes, sep='\n- ')
display(data_analysis.summary)
display(data_analysis.model_metrics)

print('External-data assets:')
for path in sorted((data_analysis.run_dir / 'tables').glob('*.csv')):
    print('-', path.name)
for path in sorted((data_analysis.run_dir / 'figures').glob('*.png')):
    print('-', path.name)


## 5. Inspect the complete CURE-Rec causal game

The game uses all six interventions, exact coalition values, exact Shapley contributions, feasibility-aware semivalue sensitivity, and Grabisch–Roubens pairwise interactions. Direct robust improvement—not a sum of Shapley lower endpoints—selects the portfolio.

In [ ]:
display(game.regions.sort_values('phi_mean', ascending=False))
display(game.interaction_table.sort_values('interaction_mean', ascending=False))

coalitions = game.coalition_table.groupby('mask', as_index=False).agg(
    lower_improvement=('improvement', 'min'),
    upper_improvement=('improvement', 'max'),
    cost=('cost', 'first'),
    interventions=('active_interventions', 'first'),
).sort_values('lower_improvement', ascending=False)
display(coalitions.head(12))


## 6. Inspect all numbered paper assets, logs, and manifests

Every CURE-Sim run generates Tables 1–8, Figures 1–8, an asset registry, per-coalition manifests, JSONL events, raw coalition values, and a deployment/explanation decision card.

In [ ]:
asset_manifest = json.loads((RUN_DIR / 'artifacts' / 'asset_manifest.json').read_text())
asset_registry = pd.DataFrame(asset_manifest)
display(asset_registry)

print('Generated CURE tables:')
for path in sorted((RUN_DIR / 'tables').glob('*.csv')):
    print('-', path.name)
print('\nGenerated CURE figures:')
for path in sorted((RUN_DIR / 'figures').glob('*.png')):
    print('-', path.name)

events = pd.DataFrame([json.loads(line) for line in (RUN_DIR / 'logs' / 'events.jsonl').read_text().splitlines()])
display(events[['timestamp_utc', 'event']].tail(20))

decision_card = json.loads((RUN_DIR / 'artifacts' / 'explanation_card.json').read_text())
decision_card


## 7. Evidence and runtime notes

- MovieLens, Coat, Yahoo! R3, and generic ratings CSVs are loaded and audited before baseline-model analysis. Their audit result controls the permitted scientific claim.
- CURE-Sim is the full causal/oracle environment in this milestone.
- Real long-horizon policy/OPE assets remain gated until an audited slate-policy log and sequential estimator are implemented.
- For a larger CURE-Sim run, change `CURE_MODE = 'full'` and rerun this notebook from the top.
- The same workflow is available from the terminal via `cure-rec full-run`.